# Demo A — Da PCAP a Grafo
**Master Cybersecurity LUISS — 29 maggio 2026**  
Antonio Formato

**Pipeline:** `tshark` → `pandas` → `networkx` → `pyvis`

**Tre viste progressive sullo stesso pcap:**
1. **Naive** — tutti i flussi, nessuna aggregazione (caos visivo voluto)
2. **Aggregata pesata** — gli hub emergono
3. **Filtrata (external destinations)** — potenziali C2/exfiltration

**Bonus didattico:** applichiamo gli stessi algoritmi visti nella teoria T2 (PageRank, connected components) al grafo costruito dal pcap. Chiusura del cerchio.

## 0. Setup
Installazione una tantum (se non già fatto):  
```bash
pip install pandas networkx pyvis
```
**Inoltre serve `tshark` nel PATH.** Su Windows: lo installi insieme a Wireshark, poi aggiungi `C:\Program Files\Wireshark` al PATH. Su macOS: `brew install wireshark`. Su Ubuntu: `apt install tshark`.

**Se non puoi/vuoi usare tshark**, salta direttamente al fallback (cella più sotto) che carica un CSV pre-estratto.

In [1]:
import subprocess
import ipaddress
from pathlib import Path

import pandas as pd
import networkx as nx
from pyvis.network import Network

In [2]:
# ───── CONFIGURAZIONE ─────
PCAP = "sample.pcap"          # ← sostituisci col path al tuo pcap
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
FLOWS_CSV = OUTPUT_DIR / "flows.csv"

## 1. Estrazione flussi con tshark
Estraiamo IP sorgente, IP destinazione e porte. Filtriamo solo i pacchetti con IP layer.

In [ ]:
def extract_flows(pcap_path, csv_path):
    """Estrae src/dst/porte da un pcap usando tshark."""
    cmd = [
        "tshark", "-r", str(pcap_path),
        "-T", "fields",
        "-e", "ip.src",
        "-e", "ip.dst",
        "-e", "tcp.dstport",
        "-e", "udp.dstport",
        "-E", "separator=,",
        "-E", "header=n",
        "-Y", "ip",   # solo pacchetti con IP
    ]
    with open(csv_path, "w") as f:
        subprocess.run(cmd, stdout=f, check=True)
    print(f"✓ Estratto in {csv_path}")

extract_flows(PCAP, FLOWS_CSV)

### Fallback: niente tshark
Se tshark non è disponibile, puoi pre-estrarre i flussi da linea di comando e mettere il CSV in `output/flows.csv`:
```bash
tshark -r sample.pcap -T fields -e ip.src -e ip.dst -e tcp.dstport -e udp.dstport -E separator=, -Y ip > output/flows.csv
```
Oppure: puoi anche generare flussi sintetici per provare il resto del notebook (cella commentata sotto).

In [ ]:
# # Dati sintetici di prova — decommentare per testare senza pcap
# import random
# random.seed(42)
# internal_hosts = [f"10.0.0.{i}" for i in range(1, 8)]
# external_hosts = [f"203.0.113.{i}" for i in [10, 25, 42, 77]] + ["8.8.8.8"]
# c2 = "203.0.113.42"  # nostro "C2" finto: beaconing
# rows = []
# for _ in range(500):
#     src = random.choice(internal_hosts)
#     dst = random.choice(external_hosts + internal_hosts)
#     dport = random.choice([80, 443, 53, 22, 3389])
#     rows.append((src, dst, dport, ""))
# # beaconing fitto verso il C2
# for _ in range(200):
#     rows.append((random.choice(internal_hosts), c2, 443, ""))
# pd.DataFrame(rows, columns=["src","dst","tcp_dport","udp_dport"]).to_csv(FLOWS_CSV, index=False, header=False)
# print("✓ CSV sintetico creato")

## 2. Caricamento e ispezione

In [3]:
df = pd.read_csv(FLOWS_CSV, names=["src", "dst", "tcp_dport", "udp_dport"], dtype=str)
df["dport"] = df["tcp_dport"].fillna(df["udp_dport"])
df = df.dropna(subset=["src", "dst"])

print(f"Flussi totali:      {len(df):,}")
print(f"IP src unici:       {df['src'].nunique()}")
print(f"IP dst unici:       {df['dst'].nunique()}")
print(f"Porte dst uniche:   {df['dport'].nunique()}")
df.head()

Flussi totali:      20,555
IP src unici:       38
IP dst unici:       42
Porte dst uniche:   181


,src,dst,tcp_dport,udp_dport,dport
0,10.5.11.101,10.5.11.1,NaN,53,53
1,10.5.11.101,10.5.11.1,NaN,53,53
2,10.5.11.1,10.5.11.101,NaN,62429,62429
3,10.5.11.1,10.5.11.101,NaN,50153,50153
4,10.5.11.101,142.250.114.113,443,NaN,443


## 3. Vista 1 — Grafo ingenuo
Tutti i flussi, senza aggregazione. **Risultato: caos visivo.**  
*Pedagogicamente serve a mostrare perché aggregare è necessario.*

In [4]:
G1 = nx.from_pandas_edgelist(df, "src", "dst", create_using=nx.DiGraph())
print(f"Nodi: {G1.number_of_nodes()}, Archi: {G1.number_of_edges()}")

net1 = Network(notebook=False, directed=True, height="650px", width="100%",
               bgcolor="#1a1a1a", font_color="white")
net1.from_nx(G1)
net1.write_html(str(OUTPUT_DIR / "view1_naive.html"), open_browser=False, notebook=False)
print(f"→ {OUTPUT_DIR / 'view1_naive.html'}")

Nodi: 42, Archi: 78
→ output\view1_naive.html


## 4. Vista 2 — Grafo aggregato e pesato
Stessi nodi. **L'arco src→dst ha peso = numero di flussi.** Gli hub emergono.

In [5]:
weighted = df.groupby(["src", "dst"]).size().reset_index(name="weight")
G2 = nx.from_pandas_edgelist(weighted, "src", "dst",
                              edge_attr="weight", create_using=nx.DiGraph())

# Dimensione nodo ∝ volume traffico totale (sqrt per non esplodere)
out_volume = weighted.groupby("src")["weight"].sum().to_dict()
in_volume = weighted.groupby("dst")["weight"].sum().to_dict()
for n in G2.nodes():
    total = out_volume.get(n, 0) + in_volume.get(n, 0)
    G2.nodes[n]["size"] = 8 + (total ** 0.5) * 1.5
    G2.nodes[n]["label"] = n
    G2.nodes[n]["title"] = f"{n}\nVolume: {total} flussi"

net2 = Network(notebook=False, directed=True, height="650px", width="100%",
               bgcolor="#1a1a1a", font_color="white")
net2.from_nx(G2)
net2.write_html(str(OUTPUT_DIR / "view2_weighted.html"), open_browser=False, notebook=False)
print(f"→ {OUTPUT_DIR / 'view2_weighted.html'}")

→ output\view2_weighted.html


## 5. Vista 3 — Solo destinazioni esterne
Filtriamo: **flussi da host interno (RFC1918) verso destinazioni pubbliche.**  
Quello che resta sono i potenziali target di C2 o exfiltration.

In [6]:
# Solo RFC1918 conta come "interno". La is_private di Python include anche
# range documentation/reserved (es. 203.0.113.0/24) che qui non vogliamo.
RFC1918 = [ipaddress.ip_network(n) for n in
           ("10.0.0.0/8", "172.16.0.0/12", "192.168.0.0/16")]

def is_internal(ip):
    try:
        addr = ipaddress.ip_address(ip)
        return any(addr in net for net in RFC1918)
    except ValueError:
        return False

ext = weighted[
    weighted["src"].apply(is_internal) &
    ~weighted["dst"].apply(is_internal)
]
print(f"Host interni che parlano verso l'esterno: {ext['src'].nunique()}")
print(f"Destinazioni esterne distinte:            {ext['dst'].nunique()}")

G3 = nx.from_pandas_edgelist(ext, "src", "dst",
                              edge_attr="weight", create_using=nx.DiGraph())
for n in G3.nodes():
    internal = is_internal(n)
    G3.nodes[n]["color"] = "#4ecdc4" if internal else "#ff6b6b"
    G3.nodes[n]["label"] = n
    G3.nodes[n]["size"] = 15 if internal else 20

# spessore arco proporzionale al peso
for u, v, d in G3.edges(data=True):
    d["width"] = 1 + d.get("weight", 1) ** 0.5

net3 = Network(notebook=False, directed=True, height="650px", width="100%",
               bgcolor="#1a1a1a", font_color="white")
net3.from_nx(G3)
net3.write_html(str(OUTPUT_DIR / "view3_external.html"), open_browser=False, notebook=False)
print(f"→ {OUTPUT_DIR / 'view3_external.html'}")

Host interni che parlano verso l'esterno: 1
Destinazioni esterne distinte:            39
→ output\view3_external.html


## 6. Bonus — Stessi algoritmi della teoria, applicati live
Nella parte teorica T2 abbiamo visto PageRank, community detection, connected components.  
**Ora li applichiamo al grafo costruito dal pcap.**  
È il momento in cui la teoria diventa visibile.

### 6.1 PageRank — quali IP sono "centrali" nel traffico?

In [7]:
pr = nx.pagerank(G2, weight="weight")
top10 = sorted(pr.items(), key=lambda x: -x[1])[:10]

print("Top 10 nodi per PageRank:\n")
print(f"{'IP':<20} {'Score':<10} {'Tipo'}")
print("-" * 45)
for ip, score in top10:
    tipo = "interno" if is_internal(ip) else "ESTERNO"
    print(f"{ip:<20} {score:<10.4f} {tipo}")

Top 10 nodi per PageRank:

IP                   Score      Tipo
---------------------------------------------
10.5.11.101          0.4368     interno
10.5.11.1            0.0137     interno
10.5.11.255          0.0137     interno
104.16.79.73         0.0137     ESTERNO
104.21.26.126        0.0137     ESTERNO
104.21.41.172        0.0137     ESTERNO
104.21.41.99         0.0137     ESTERNO
104.21.89.164        0.0137     ESTERNO
142.250.114.113      0.0137     ESTERNO
142.251.116.94       0.0137     ESTERNO


### 6.2 Connected components — cluster di comunicazione

In [8]:
G2_undirected = G2.to_undirected()
components = list(nx.connected_components(G2_undirected))
components.sort(key=len, reverse=True)

print(f"Numero di componenti connesse: {len(components)}\n")
for i, comp in enumerate(components[:5], 1):
    print(f"Componente #{i}: {len(comp)} nodi  — esempio: {list(comp)[:3]}")

Numero di componenti connesse: 1

Componente #1: 42 nodi  — esempio: ['17.253.12.37', '173.194.208.94', '104.21.26.126']


### 6.3 Visualizzazione finale con PageRank come colore

In [9]:
# Color encoding: PageRank → intensità
pr_max = max(pr.values()) if pr else 1
G_final = G2.copy()
for n in G_final.nodes():
    score = pr.get(n, 0) / pr_max
    # rosso più intenso = PageRank più alto
    red = int(80 + 175 * score)
    G_final.nodes[n]["color"] = f"rgb({red},{100 - int(50*score)},{100 - int(50*score)})"
    G_final.nodes[n]["title"] = f"{n}\nPageRank: {pr.get(n, 0):.4f}"

net_final = Network(notebook=False, directed=True, height="700px", width="100%",
                    bgcolor="#1a1a1a", font_color="white")
net_final.from_nx(G_final)
net_final.write_html(str(OUTPUT_DIR / "view_bonus_pagerank.html"),
                     open_browser=False, notebook=False)
print(f"→ {OUTPUT_DIR / 'view_bonus_pagerank.html'}")

→ output\view_bonus_pagerank.html


## 7. Recap — cosa hai mostrato in aula

1. **Da un pcap di un singolo cliente/lab → un grafo interrogabile** in <30 righe di Python  
2. **Tre rappresentazioni progressive** dello stesso oggetto matematico (lo stesso pcap)  
3. **Gli algoritmi astratti della teoria** (PageRank, connected components) **diventano concreti** sul traffico reale  
4. **Bridge naturale verso BloodHound** (demo successiva): stesso paradigma, dominio diverso (identità AD invece di pacchetti IP)